In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [ ]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

In [ ]:
df = pd.read_csv(INTERIM_DIR / "atp_matches_1990_onward_base.csv")
df.shape

In [ ]:
df["tourney_date"] = pd.to_datetime(df["tourney_date"], format = "%Y%m%d")
df = df.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop = True)
df.head()

In [ ]:
def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a)/ 400.0))

In [ ]:
ratings = defaultdict(lambda: 1500.0)

winner_elo_before = []
loser_elo_before = []
winner_expected = []
loser_expected = []

K = 32.0

for row in df.itertuples(index = False) :
    w_id = row.winner_id
    l_id = row.loser_id

    w_rating = ratings[w_id]
    l_rating = ratings[l_id]

    w_exp = expected_score(w_rating, l_rating)
    l_exp = expected_score(l_rating, w_rating)

    winner_elo_before.append(w_rating)
    loser_elo_before.append(l_rating)
    winner_expected.append(w_exp)
    loser_expected.append(l_exp)

    ratings[w_id] = w_rating + K * (1.0 - w_exp)
    ratings[l_id] = l_rating + K * (0.0 - l_exp)

In [ ]:
df["winner_elo"] = winner_elo_before
df["loser_elo"] = loser_elo_before
df["winner_exp"] = winner_expected
df["loser_exp"] = loser_expected

df[[
    "tourney_date",
    "winner_name",
    "loser_name",
    "winner_elo",
    "loser_elo",
    "winner_exp"
]].head(10)

In [ ]:
df.to_csv(PROCESSED_DIR / "atp_matches_1990_onward_with_elo.csv", index = False)